# Practicum 4 - AI-gestuurde PyGame-game

Deze notebook maakt de benodigde bestanden voor een simpele PyGame-game met dataverzameling, training van een neuraal netwerk en AI-play mode. PyGame werkt meestal beter als `.py` script vanuit de terminal dan direct in Jupyter.

In [1]:
# Installatie indien nodig:
# !pip install pygame pandas numpy matplotlib scikit-learn tensorflow joblib openpyxl

from pathlib import Path
PROJECT_DIR = Path.cwd() / "practicum4_ai_game"
PROJECT_DIR.mkdir(exist_ok=True)
(PROJECT_DIR / "data").mkdir(exist_ok=True)
(PROJECT_DIR / "models").mkdir(exist_ok=True)
(PROJECT_DIR / "results").mkdir(exist_ok=True)
print("Projectmap:", PROJECT_DIR)

Projectmap: /Users/cmok/Documents/GitHub/DEAI-SE4/NN/practicum4_ai_game


## 1. Game + dataverzameling

Run deze cel om `game_collect_data.py` te maken. Speel daarna zelf om trainingsdata te verzamelen.

In [2]:
game_code = 'import csv\nimport os\nimport random\nimport pygame\n\nWIDTH, HEIGHT = 500, 600\nPLAYER_W, PLAYER_H = 50, 20\nROCK_SIZE = 30\nPLAYER_SPEED = 6\nFPS = 60\nDATA_PATH = os.path.join("data", "training_data.csv")\n\nos.makedirs("data", exist_ok=True)\n\nclass Rock:\n    def __init__(self):\n        self.x = random.randint(0, WIDTH - ROCK_SIZE)\n        self.y = -ROCK_SIZE\n        self.speed = random.randint(4, 8)\n    def update(self):\n        self.y += self.speed\n    def reset(self):\n        self.x = random.randint(0, WIDTH - ROCK_SIZE)\n        self.y = -ROCK_SIZE\n        self.speed = random.randint(4, 8)\n\ndef normalized_features(player_x, rock):\n    player_center = player_x + PLAYER_W / 2\n    rock_center = rock.x + ROCK_SIZE / 2\n    dx = rock_center - player_center\n    return [player_center / WIDTH, rock_center / WIDTH, rock.y / HEIGHT, dx / WIDTH, rock.speed / 10]\n\ndef save_row(features, action):\n    file_exists = os.path.exists(DATA_PATH)\n    with open(DATA_PATH, "a", newline="") as f:\n        writer = csv.writer(f)\n        if not file_exists:\n            writer.writerow(["player_x", "rock_x", "rock_y", "dx", "rock_speed", "action"])\n        writer.writerow(features + [action])\n\ndef main():\n    pygame.init()\n    screen = pygame.display.set_mode((WIDTH, HEIGHT))\n    pygame.display.set_caption("Practicum 4 - Human data collection")\n    clock = pygame.time.Clock()\n    font = pygame.font.SysFont(None, 28)\n    player_x = WIDTH // 2 - PLAYER_W // 2\n    rock = Rock()\n    score = 0\n    running = True\n    while running:\n        clock.tick(FPS)\n        action = 1\n        for event in pygame.event.get():\n            if event.type == pygame.QUIT:\n                running = False\n        keys = pygame.key.get_pressed()\n        if keys[pygame.K_ESCAPE]:\n            running = False\n        if keys[pygame.K_LEFT]:\n            player_x -= PLAYER_SPEED; action = 0\n        elif keys[pygame.K_RIGHT]:\n            player_x += PLAYER_SPEED; action = 2\n        player_x = max(0, min(WIDTH - PLAYER_W, player_x))\n        rock.update()\n        save_row(normalized_features(player_x, rock), action)\n        player_rect = pygame.Rect(player_x, HEIGHT - 60, PLAYER_W, PLAYER_H)\n        rock_rect = pygame.Rect(rock.x, rock.y, ROCK_SIZE, ROCK_SIZE)\n        if player_rect.colliderect(rock_rect):\n            print("Game over! Score:", score); running = False\n        if rock.y > HEIGHT:\n            score += 1; rock.reset()\n        screen.fill((15, 18, 28))\n        pygame.draw.rect(screen, (80, 180, 255), player_rect)\n        pygame.draw.rect(screen, (180, 180, 180), rock_rect)\n        text = font.render(f"Human mode | Score: {score} | Rows saved", True, (255,255,255))\n        screen.blit(text, (10, 10))\n        pygame.display.flip()\n    pygame.quit()\n\nif __name__ == "__main__":\n    main()\n'
(PROJECT_DIR / "game_collect_data.py").write_text(game_code, encoding="utf-8")
print("Aangemaakt:", PROJECT_DIR / "game_collect_data.py")

Aangemaakt: /Users/cmok/Documents/GitHub/DEAI-SE4/NN/practicum4_ai_game/game_collect_data.py


In [3]:
print("Start human game met:")
print(f"cd {PROJECT_DIR}")
print("python game_collect_data.py")

Start human game met:
cd /Users/cmok/Documents/GitHub/DEAI-SE4/NN/practicum4_ai_game
python game_collect_data.py


## 2. Data controleren

In [16]:
import pandas as pd
csv_path = PROJECT_DIR / "data" / "training_data.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    display(df.head())
    print("Aantal rijen:", len(df))
    print(df["action"].value_counts().sort_index())
else:
    print("Nog geen data gevonden. Speel eerst de game_collect_data.py.")

Nog geen data gevonden. Speel eerst de game_collect_data.py.


## 3. Model trainen

Run deze cel om `train_model.py` te maken. Dit script voert 30 experimenten uit en bewaart resultaten.

In [17]:
train_code = 'import os\nimport joblib\nimport pandas as pd\nimport numpy as np\nimport tensorflow as tf\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.metrics import accuracy_score, f1_score\n\nDATA_PATH = os.path.join("data", "training_data.csv")\nMODEL_DIR = "models"\nRESULT_DIR = "results"\nos.makedirs(MODEL_DIR, exist_ok=True)\nos.makedirs(RESULT_DIR, exist_ok=True)\n\nif not os.path.exists(DATA_PATH):\n    raise FileNotFoundError("Geen training_data.csv gevonden. Speel eerst de game in human mode.")\n\ndf = pd.read_csv(DATA_PATH).dropna()\nX = df[["player_x", "rock_x", "rock_y", "dx", "rock_speed"]].values\ny = df["action"].values\n\nscaler = StandardScaler()\nX_scaled = scaler.fit_transform(X)\njoblib.dump(scaler, os.path.join(MODEL_DIR, "scaler.joblib"))\n\nX_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.25, random_state=42, stratify=y)\n\nbase_experiments = [\n    {"layers": [16], "epochs": 10, "lr": 0.001, "batch": 32},\n    {"layers": [32], "epochs": 10, "lr": 0.001, "batch": 32},\n    {"layers": [64], "epochs": 15, "lr": 0.001, "batch": 32},\n    {"layers": [32, 16], "epochs": 15, "lr": 0.001, "batch": 32},\n    {"layers": [64, 32], "epochs": 20, "lr": 0.001, "batch": 32},\n    {"layers": [128, 64], "epochs": 20, "lr": 0.001, "batch": 64},\n    {"layers": [32], "epochs": 20, "lr": 0.01, "batch": 32},\n    {"layers": [32], "epochs": 20, "lr": 0.0001, "batch": 32},\n    {"layers": [64, 32, 16], "epochs": 20, "lr": 0.001, "batch": 32},\n    {"layers": [128, 64, 32], "epochs": 25, "lr": 0.001, "batch": 64},\n]\nexperiments = []\nfor base in base_experiments:\n    for epochs in [5, 10, 20]:\n        copy = dict(base); copy["epochs"] = epochs; experiments.append(copy)\nexperiments = experiments[:30]\n\ndef build_model(layers, lr):\n    model = tf.keras.Sequential([tf.keras.layers.Input(shape=(X_train.shape[1],))])\n    for nodes in layers:\n        model.add(tf.keras.layers.Dense(nodes, activation="relu"))\n    model.add(tf.keras.layers.Dense(3, activation="softmax"))\n    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])\n    return model\n\nresults = []\nbest_acc = -1\nbest_model = None\nfor i, exp in enumerate(experiments, start=1):\n    print(f"Experiment {i}/{len(experiments)}", exp)\n    model = build_model(exp["layers"], exp["lr"])\n    history = model.fit(X_train, y_train, validation_split=0.2, epochs=exp["epochs"], batch_size=exp["batch"], verbose=0)\n    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)\n    test_acc = accuracy_score(y_test, y_pred)\n    f1 = f1_score(y_test, y_pred, average="weighted")\n    row = {"experiment": i, "layers": str(exp["layers"]), "epochs": exp["epochs"], "learning_rate": exp["lr"], "batch_size": exp["batch"], "dataset_size": len(df), "train_accuracy": history.history["accuracy"][-1], "validation_accuracy": history.history["val_accuracy"][-1], "test_accuracy": test_acc, "f1_score": f1}\n    results.append(row)\n    if test_acc > best_acc:\n        best_acc = test_acc; best_model = model\n\nresults_df = pd.DataFrame(results)\nresults_df.to_csv(os.path.join(RESULT_DIR, "experiment_results.csv"), index=False)\nresults_df.to_excel(os.path.join(RESULT_DIR, "experiment_results.xlsx"), index=False)\nbest_model.save(os.path.join(MODEL_DIR, "best_model.keras"))\nprint("Beste test accuracy:", best_acc)\nprint(results_df.sort_values("test_accuracy", ascending=False).head())\n'
(PROJECT_DIR / "train_model.py").write_text(train_code, encoding="utf-8")
print("Aangemaakt:", PROJECT_DIR / "train_model.py")

Aangemaakt: /Users/cmok/Documents/GitHub/DEAI-SE4/NN/practicum4_ai_game/train_model.py


In [18]:
print("Train model met:")
print(f"cd {PROJECT_DIR}")
print("python train_model.py")

Train model met:
cd /Users/cmok/Documents/GitHub/DEAI-SE4/NN/practicum4_ai_game
python train_model.py


## 4. AI laten spelen

In [19]:
ai_code = 'import os\nimport random\nimport joblib\nimport numpy as np\nimport pygame\nimport tensorflow as tf\n\nWIDTH, HEIGHT = 500, 600\nPLAYER_W, PLAYER_H = 50, 20\nROCK_SIZE = 30\nPLAYER_SPEED = 6\nFPS = 60\nMODEL_PATH = os.path.join("models", "best_model.keras")\nSCALER_PATH = os.path.join("models", "scaler.joblib")\n\nclass Rock:\n    def __init__(self):\n        self.x = random.randint(0, WIDTH - ROCK_SIZE)\n        self.y = -ROCK_SIZE\n        self.speed = random.randint(4, 8)\n    def update(self):\n        self.y += self.speed\n    def reset(self):\n        self.x = random.randint(0, WIDTH - ROCK_SIZE)\n        self.y = -ROCK_SIZE\n        self.speed = random.randint(4, 8)\n\ndef features(player_x, rock):\n    player_center = player_x + PLAYER_W / 2\n    rock_center = rock.x + ROCK_SIZE / 2\n    dx = rock_center - player_center\n    return np.array([[player_center / WIDTH, rock_center / WIDTH, rock.y / HEIGHT, dx / WIDTH, rock.speed / 10]])\n\ndef main():\n    if not os.path.exists(MODEL_PATH):\n        raise FileNotFoundError("Geen getraind model gevonden. Run eerst train_model.py")\n    model = tf.keras.models.load_model(MODEL_PATH)\n    scaler = joblib.load(SCALER_PATH)\n    pygame.init()\n    screen = pygame.display.set_mode((WIDTH, HEIGHT))\n    pygame.display.set_caption("Practicum 4 - AI mode")\n    clock = pygame.time.Clock()\n    font = pygame.font.SysFont(None, 28)\n    player_x = WIDTH // 2 - PLAYER_W // 2\n    rock = Rock()\n    score = 0\n    running = True\n    while running:\n        clock.tick(FPS)\n        for event in pygame.event.get():\n            if event.type == pygame.QUIT:\n                running = False\n        keys = pygame.key.get_pressed()\n        if keys[pygame.K_ESCAPE]:\n            running = False\n        X = scaler.transform(features(player_x, rock))\n        action = int(np.argmax(model.predict(X, verbose=0)[0]))\n        if action == 0: player_x -= PLAYER_SPEED\n        elif action == 2: player_x += PLAYER_SPEED\n        player_x = max(0, min(WIDTH - PLAYER_W, player_x))\n        rock.update()\n        player_rect = pygame.Rect(player_x, HEIGHT - 60, PLAYER_W, PLAYER_H)\n        rock_rect = pygame.Rect(rock.x, rock.y, ROCK_SIZE, ROCK_SIZE)\n        if player_rect.colliderect(rock_rect):\n            print("AI game over! Score:", score); running = False\n        if rock.y > HEIGHT:\n            score += 1; rock.reset()\n        screen.fill((10, 20, 18))\n        pygame.draw.rect(screen, (0, 220, 120), player_rect)\n        pygame.draw.rect(screen, (210, 210, 210), rock_rect)\n        label = ["links", "stil", "rechts"][action]\n        text = font.render(f"AI mode | Score: {score} | Actie: {label}", True, (255,255,255))\n        screen.blit(text, (10, 10))\n        pygame.display.flip()\n    pygame.quit()\n\nif __name__ == "__main__":\n    main()\n'
(PROJECT_DIR / "play_ai.py").write_text(ai_code, encoding="utf-8")
print("Aangemaakt:", PROJECT_DIR / "play_ai.py")

Aangemaakt: /Users/cmok/Documents/GitHub/DEAI-SE4/NN/practicum4_ai_game/play_ai.py


In [20]:
print("Start AI game met:")
print(f"cd {PROJECT_DIR}")
print("python play_ai.py")

Start AI game met:
cd /Users/cmok/Documents/GitHub/DEAI-SE4/NN/practicum4_ai_game
python play_ai.py


## 5. Resultaten en grafieken

Run deze cel nadat `train_model.py` is uitgevoerd.

In [24]:
import pandas as pd
import matplotlib.pyplot as plt
results_path = PROJECT_DIR / "results" / "experiment_results.csv"
if results_path.exists():
    results = pd.read_csv(results_path)
    display(results.sort_values("test_accuracy", ascending=False).head(10))
    plt.figure(figsize=(12, 5))
    plt.bar(results["experiment"], results["test_accuracy"])
    plt.xlabel("Experiment")
    plt.ylabel("Test accuracy")
    plt.title("Test accuracy per experiment")
    plt.show()
    plt.figure(figsize=(12, 5))
    plt.plot(results["experiment"], results["train_accuracy"], marker="o", label="Train accuracy")
    plt.plot(results["experiment"], results["test_accuracy"], marker="o", label="Test accuracy")
    plt.xlabel("Experiment")
    plt.ylabel("Accuracy")
    plt.title("Train vs test accuracy")
    plt.legend()
    plt.show()
else:
    print("Nog geen experiment_results.csv gevonden. Train eerst het model.")

Nog geen experiment_results.csv gevonden. Train eerst het model.


## 6. Uitleg voor verslag

- De input van het model bestaat niet uit pixels, maar uit compacte numerieke features.
- De output is een actieklasse: links, stilstaan of rechts.
- Het model leert supervised learning: inputvoorbeelden met labels uit menselijk speelgedrag.
- De loss is sparse categorical crossentropy omdat er drie klassen zijn.
- Een hoge test accuracy betekent niet automatisch dat de AI goed speelt; meet ook game-score of overlevingstijd.